# Step C: train all five models, three seeds each

Protocol fixed in `docs/DECISIONS.md` **before** training (DEC-005 range, DEC-006 hypotheses, DEC-007 protocol):

- **M0** trains on clean scans; **M1, M4, M5, M6** see one of 13 degraded versions of each slice (k-space noise SNR 8/5/3 × resolution 1.0/0.5 × slice 1/5 mm, plus clean)
- all brain slices (tumor-free included), 20 epochs, full-volume 3D validation under clean / SNR 5 / worst
- 5 models × seeds 0, 1, 2 = **15 runs**, about 30-45 min each on an L4

**How to run it:** *Runtime → Change runtime type → L4 GPU*, then *Run all*. Each finished run is skipped automatically and an interrupted one resumes from its last epoch, so if Colab stops, just *Run all* again (setup takes ~30 min per new session). Runs go seed by seed (all 5 models on seed 0 first), so usable results arrive early.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

name, memory.total [MiB]
Tesla T4, 15360 MiB
overlay         113G   43G   71G  38% /


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ---- Configuration ----
ZIP    = '/content/drive/MyDrive/Argus/archive.zip'
OUT    = '/content/drive/MyDrive/Argus/train_v3'     # Step C checkpoints + logs (resumable)
WORK   = '/content/cache'
MODELS = ['M0', 'M1', 'M4', 'M5', 'M6']
SEEDS  = [0, 1, 2]
EPOCHS = 20
REPO, BRANCH = 'https://github.com/joshykoshy/Argus.git', 'matlab-analysis'
import os; os.makedirs(OUT, exist_ok=True)

In [4]:
import os
if os.path.isdir('/content/Argus'):
    !git -C /content/Argus pull -q
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Argus
%cd /content/Argus
!pip -q install nibabel

/content/Argus


In [5]:
# Extract the 258 training + 37 validation patients (~25 GB, 10-20 min). Skipped if already prepared.
import json, zipfile, glob
split = json.load(open('data/splits/split_v1.json'))
need = split['train'] + split['val']
prepared = all(os.path.exists(f'{WORK}/{p}_mu.npy') for p in need)
if not prepared and not glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True):
    with zipfile.ZipFile(ZIP) as z:
        names = [n for n in z.namelist() if any(f'/{p}/' in n for p in need) or n.endswith('/name_mapping.csv')]
        z.extractall('/content/brats', members=names)
RAW = (glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True) or [''])[0]
print('prepared already' if prepared else f'{RAW}: raw data ready')

/content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData: raw data ready


In [6]:
# Preprocess into memory-mappable arrays (~15-25 min, resumable), then free the raw files
if not prepared:
    !python -m training.prepare --raw_dir {RAW} --work {WORK}
    !rm -rf /content/brats
!du -sh {WORK}

prepared 20/295
prepared 40/295
prepared 60/295
prepared 80/295
prepared 100/295
prepared 120/295
prepared 140/295
prepared 160/295
prepared 180/295
prepared 200/295
prepared 220/295
prepared 240/295
prepared 260/295
prepared 280/295
prepared 295/295
PREPARE DONE
16G	/content/cache


In [7]:
# Smoke test on the hardest path (M4 = dual-stream + degradation): 1 tiny epoch, 2 validation patients
!rm -rf /content/smoke
!python -m training.train_v2 --model M4 --work {WORK} --out /content/smoke --epochs 1 --slices_per_epoch 640 --val_patients 2

M4 seed 0: 34999 brain slices from 258 patients, degradation augmentation ON (13 DEC-005 conditions), device cuda
epoch 1: loss 1.60816, val mean 3D Dice 0.0382 (clean / SNR5 / worst: 0.0431 / 0.0373 / 0.0342), false tumor 267.8 cm3, 13 s
TRAINING DONE M4 seed 0. Best mean 3D Dice 0.0382 -> /content/smoke/M4/seed_0/best_model.pt


In [ ]:
# All 15 runs, seed by seed. Finished runs are skipped, an interrupted one resumes.
for seed in SEEDS:
    for model in MODELS:
        !python -m training.train_v2 --model {model} --seed {seed} --epochs {EPOCHS} --work {WORK} --out {OUT}

M0 seed 0: 34999 brain slices from 258 patients, degradation augmentation OFF (clean), device cuda
Resuming from epoch 21 (best so far 0.6797)
TRAINING DONE M0 seed 0. Best mean 3D Dice 0.6797 -> /content/drive/MyDrive/Argus/train_v3/M0/seed_0/best_model.pt
M1 seed 0: 34999 brain slices from 258 patients, degradation augmentation ON (13 DEC-005 conditions), device cuda
Resuming from epoch 20 (best so far 0.7925)
  epoch 20 batch 100/375 loss 0.1438 (2.7 batch/s)
  epoch 20 batch 200/375 loss 0.1400 (2.7 batch/s)
  epoch 20 batch 300/375 loss 0.1381 (2.6 batch/s)
epoch 20: loss 0.13857, val mean 3D Dice 0.8009 (clean / SNR5 / worst: 0.8249 / 0.8252 / 0.7526), false tumor 0.2 cm3, 143 s
TRAINING DONE M1 seed 0. Best mean 3D Dice 0.8009 -> /content/drive/MyDrive/Argus/train_v3/M1/seed_0/best_model.pt
M4 seed 0: 34999 brain slices from 258 patients, degradation augmentation ON (13 DEC-005 conditions), device cuda
  epoch 1 batch 100/375 loss 1.5791 (2.5 batch/s)


In [ ]:
# Progress / results table: best validation epoch of every finished run
import pandas as pd, glob
rows = []
for f in sorted(glob.glob(f'{OUT}/*/seed_*/log.csv')):
    log = pd.read_csv(f).dropna(subset=['val_mean'])
    if len(log):
        b = log.loc[log.val_mean.idxmax()]
        rows.append(dict(model=f.split('/')[-3], seed=f.split('/')[-2], epochs_done=int(pd.read_csv(f).epoch.max()),
                         best_epoch=int(b.epoch), clean=b['val_clean'], snr5=b['val_kspace_snr5_r1.0_t1'],
                         worst=b['val_kspace_snr3_r0.5_t5'], mean=b.val_mean, false_tumor_cm3=b.val_false_tumor_cm3))
R = pd.DataFrame(rows); print(R.to_string(index=False))
R.to_csv(f'{OUT}/stepC_summary.csv', index=False)

**When all 15 are done** (or whenever you want a progress check): just say so. `stepC_summary.csv` and every run's `log.csv` are in `MyDrive/Argus/train_v3/`. These are **validation** numbers used to pick each model's best epoch; the paper's results come from the 74 untouched test patients in the next step (Step D).